# Point Cloud to CAD-sequence

In this notebook the complete interactive pipeline for encoding point clouds into a latent space, from which DeepCAD decodes a CAD-sequence.

In [12]:
import os
import sys
import importlib

import torch

sys.path.append("..")
sys.path.append("../code")
from dataset import PointCloudEmbeddingSequenceDataset
from models.DeepCAD.config.configAE import ConfigAE
from models.DeepCAD.trainer.trainerAE import TrainerAE

### Variables

Store the models in ```experiments```, a results directory will be created for each respective model.

In [2]:
model_name = "best"

### Constants

In [3]:
model_path = os.path.join("experiments", model_name) + ".pth"
results_dir = os.path.join("experiments", model_name + "_results")
if not os.path.exists(results_dir):
    os.mkdir(results_dir)
latent_dim = 256

### Create and load pre-trained PointNet++

In [4]:
def inplace_relu(m):
    classname = m.__class__.__name__
    if classname.find('ReLU') != -1:
        m.inplace=True

sys.path.append(os.path.join('..', 'models','Pointnet_Pointnet2_pytorch', 'models'))
model = importlib.import_module('pointnet2_cls_ssg')
classifier = model.get_model(latent_dim, normal_channel=False)
criterion = model.get_loss_mse()
classifier.apply(inplace_relu)

saved_model = torch.load(model_path, map_location=torch.device('cpu'), weights_only=True)
state_dict = saved_model['model_state_dict']
if 'module.' in next(iter(state_dict)):
    monitor.log_and_print("Model was saved wrapped in nn.DataParallel.\nRemoving 'module.' from state dict.")
    state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}#
classifier.eval()
classifier.load_state_dict(state_dict)

<All keys matched successfully>

### Create and load pre-trained DeepCAD

In [6]:
cfg = ConfigAE('test', model_path="../data/latent")
tr_agent = TrainerAE(cfg)
tr_agent.net.eval()
tr_agent.load_ckpt(cfg.ckpt)

Loading checkpoint from /Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/data/latent/pretrained/model/ckpt_epoch1000.pth ...


### Load data

### TODO 
- enable batch loading without dataloader
- make loss work for single examples

In [53]:
dataset = PointCloudEmbeddingSequenceDataset("../data", 'test')
pc, lat_rep, cad_seq = dataset[3]

Loading test dataset 

Number of samples that should be in the test set: 8052
Files on disk: 8038 --> There are 14 missing point cloud files in the test set.

Checking latent representation:
All latent represenations are valid.

--- DONE ---



In [54]:
running_loss = 0.0

lat_rep = lat_rep.unsqueeze(0)
pc = pc.unsqueeze(0)
pc = pc.transpose(2, 1)
pred, _ = classifier(pc)
loss = criterion(pred, lat_rep)
running_loss += loss.detach().item()
print(f"Avg. MSE-Loss: {running_loss:.5f}")

Avg. MSE-Loss: 0.09636


In [55]:
pred = pred.unsqueeze(0)
print(pred.shape)

torch.Size([1, 256])


In [56]:
output = tr_agent.decode(pred)

RuntimeError: mat1 and mat2 shapes cannot be multiplied (256x1 and 256x256)

In [49]:
print(output.keys())

dict_keys(['command_logits', 'args_logits'])


In [51]:
print(output['command_logits'].shape)

torch.Size([1, 60, 6])


### Gedanken

- ich sollte hier ein Ordner haben in den ich das zu benutzende model setze
- dort werden auch die ergebnisse/visualisierungen abgespeichert